# 🎯 Module 15: NLP with Deep Learning

> **Stage:** Deep Learning  
> **Level:** Intermediate → Job-Ready  
> **Module:** NLP with Deep Learning  
> **Framework:** PyTorch  
> **Date:** 15 June 2026  
> **Previous:** 14 — Computer Vision with Deep Learning  
> **Next:** 16 — Model Deployment

---

**Natural Language Processing (NLP)** is the field of AI that enables computers to process and understand human language.

Deep learning lets models **learn useful language representations directly from text** — the same way CNNs learn features from images.

In this notebook we'll cover:

- Why text is different from images
- **Tokenization** (word-level and subword)
- **Embeddings**
- **Sequence modeling** (RNN / LSTM / GRU)
- **Attention** and Transformers
- **Padding** and **attention masks**
- **Sequence-to-sequence** architectures
- **Evaluation metrics** for classification and generation
- A full **sentiment classification** project

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what NLP is and why it matters
- Convert raw text into **token IDs** and **embeddings**
- Distinguish **word-level** vs **subword** tokenization
- Use PyTorch's `nn.Embedding` correctly
- Model sequences with **LSTM / GRU**
- Explain how **attention** solves context problems
- Handle **variable-length sequences** with **padding** and **attention masks**
- Distinguish **classification** from **sequence-to-sequence** tasks
- Use appropriate **metrics** (accuracy, F1, BLEU, ROUGE)
- Build and train a **sentiment classifier** in PyTorch

---
## ⚙️ 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re, time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 🧠 1. What Is NLP?

**Natural Language Processing (NLP)** is the field of AI that enables computers to process and understand human language.

### Common applications

- Sentiment analysis
- Text classification
- Spam detection
- Named entity recognition (NER)
- Machine translation
- Text generation
- Question answering
- Search

### Typical pipeline

```text
Text
 ↓
Tokenization
 ↓
Numerical Representation
 ↓
Neural Network
 ↓
Prediction
```

---
## 🔤 2. Why Text Is Different

Images are naturally numerical (pixel values). Text is not — a computer cannot directly process:

```text
"I love this product"
```

A model needs a **numerical representation**.

### The transformation

```text
Text
 ↓
Tokens
 ↓
Token IDs
 ↓
Embeddings
 ↓
Neural Network
```

Each step converts language into something a neural network can process — while preserving meaning.

---
## ✂️ 3. Tokenization

**Tokenization** breaks text into smaller units.

### Word-level

```text
"I love AI"
→ ["I", "love", "AI"]
```

### Subword-level

Handles rare words and morphology:

```text
"unbelievable"
→ ["un", "believ", "able"]
```

Subword tokenization is the standard for modern Transformer models (BPE, WordPiece, SentencePiece).

### From tokens to IDs

Each token maps to an integer:

```text
"I"     → 15
"love"  → 82
"AI"    → 431
```

These IDs are then converted into vectors.

In [ ]:
# Simple whitespace tokenizer + vocabulary from scratch
texts = [
    "I love this movie",
    "I hate this movie",
    "this film is great",
    "this film is terrible",
    "absolutely wonderful acting",
    "absolutely awful acting",
]

def tokenize(text):
    return re.findall(r"[a-zA-Z']+", text.lower())

tokenized = [tokenize(t) for t in texts]
for t, toks in zip(texts, tokenized):
    print(f'{t!r:40s} → {toks}')

# Build vocabulary with special tokens
all_tokens = [tok for toks in tokenized for tok in toks]
counts = Counter(all_tokens)

PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token, _ in counts.most_common():
    vocab[token] = len(vocab)

print('\nVocabulary size:', len(vocab))
print('Sample IDs:', {k: vocab[k] for k in list(vocab)[:8]})

---
## 🧬 4. Word Embeddings

An embedding represents a token as a **dense numerical vector**:

```text
Token ID
   ↓
Embedding Layer
   ↓
[0.21, -0.44, 0.73, ...]
```

### In PyTorch

```python
embedding = nn.Embedding(num_embeddings=10000, embedding_dim=128)
```

Input shape: `[3]` (three token IDs)  
Output shape: `[3, 128]`

The embeddings are **learned during training** — the model discovers useful representations.

### Why embeddings matter

- They let the model learn **similarity** between words
- They compress a huge vocabulary into a small dense space
- They can be **pretrained** (Word2Vec, GloVe, or via Transformer tokenizers)

In [ ]:
# Demonstrate nn.Embedding
VOCAB_SIZE = 10000
EMBED_DIM  = 128

embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
ids = torch.tensor([15, 82, 431])
vectors = embedding(ids)

print('Input IDs    :', ids)
print('Input shape  :', ids.shape)
print('Output shape :', vectors.shape)
print('First vector :', vectors[0, :6].detach().numpy().round(3), '...')

---
## 🔁 5. Sequence Modeling

Words depend on their **surrounding context**:

```text
"I went to the bank to deposit money."

"I sat beside the river bank."
```

The same word has **different meaning** depending on context.

### Earlier deep-learning approaches

```text
Token 1 → RNN → h1
Token 2 → RNN → h2
Token 3 → RNN → h3
                ↓
             Prediction
```

**RNNs** process sequences one step at a time, carrying information forward in a hidden state.

**LSTMs** and **GRUs** improve long-range dependency handling by adding gating mechanisms.

You studied these in Modules 08 and 09.

---
## 🎯 6. Attention

Attention lets a model determine **which parts of a sequence are important** when processing a given token.

### Example

```text
"The animal didn't cross the road because it was tired."
```

Understanding what **"it"** refers to requires **context** — attention allows the model to look at *"the animal"* when processing *"it"*.

### Core equation

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

You studied the mechanism in Module 10.

### The evolution

```text
RNN
 ↓
LSTM / GRU
 ↓
Attention
 ↓
Transformer
```

---
## 🏷️ 7. Text Classification

One of the most common NLP tasks:

```text
"I really enjoyed this movie."
        ↓
      Model
        ↓
    Positive
```

### Architecture

```text
Text
 ↓
Tokenizer
 ↓
Embedding
 ↓
Transformer / RNN
 ↓
Classification Layer
 ↓
Class
```

For binary classification → **one logit** + `BCEWithLogitsLoss`.  
For multiple classes → **`nn.Linear(hidden_size, num_classes)`** + `CrossEntropyLoss`.

---
## 📏 8. Padding & Attention Masks

Sequences have different lengths:

```text
"I love AI"
"I love deep learning"
```

A batch requires **consistent dimensions**. Padding adds special `<PAD>` tokens:

```text
"I love AI <PAD> <PAD>"
"I love deep learning"
```

### ⚠️ The model must know `<PAD>` is not real content

An **attention mask** identifies valid tokens:

```text
Tokens:   I   love  AI    <PAD>  <PAD>
Mask:     1   1     1     0      0
```

1 = attend, 0 = ignore.

This becomes **critical** when working with Transformer models.

In [ ]:
# Demonstrate padding + masking
from torch.nn.utils.rnn import pad_sequence

seqs = [torch.tensor([1, 2, 3]),
        torch.tensor([4, 5]),
        torch.tensor([6, 7, 8, 9])]

padded = pad_sequence(seqs, batch_first=True, padding_value=0)
mask   = (padded != 0).long()

print('Padded sequences:')
print(padded)
print('\nAttention mask:')
print(mask)
print('\nShape:', padded.shape)

---
## ↔️ 9. Sequence-to-Sequence

Some NLP tasks transform **one sequence into another**:

- English → French
- Question → Answer
- Text → Summary

### Architecture

```text
Input Sequence
      ↓
    Encoder
      ↓
Representation
      ↓
    Decoder
      ↓
Output Sequence
```

Historically important. Modern NLP often uses **Transformer-based** encoder-decoder or decoder-only architectures (T5, BART, GPT).

---
## 📊 10. NLP Evaluation Metrics

### Classification

- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**
- **Confusion matrix**

### Generation / Translation

- **BLEU** — n-gram overlap with reference translations
- **ROUGE** — overlap for summarization

⚠️ Automated metrics don't fully capture **language quality**.

**Human evaluation** may still be necessary for open-ended generation tasks.

---
## 🛠️ 11. Practical Project: Sentiment Classification

We'll build an LSTM-based sentiment classifier **from scratch** (no external data download required).

### Pipeline

```text
Synthetic dataset
     ↓
Clean text
     ↓
Tokenize
     ↓
Vocabulary
     ↓
Embedding
     ↓
LSTM
     ↓
Linear classifier
     ↓
Positive / Negative
```

We'll use a synthetic dataset of positive/negative phrases so you can run it instantly.

In [ ]:
# Build a synthetic sentiment dataset
POSITIVE_TEMPLATES = [
    "i love this {noun}",
    "this {noun} is amazing",
    "absolutely wonderful {noun}",
    "best {noun} ever",
    "i really enjoyed the {noun}",
    "the {noun} was fantastic",
    "great {noun} highly recommend",
    "such a brilliant {noun}",
]
NEGATIVE_TEMPLATES = [
    "i hate this {noun}",
    "this {noun} is terrible",
    "absolutely awful {noun}",
    "worst {noun} ever",
    "i really disliked the {noun}",
    "the {noun} was horrible",
    "bad {noun} do not recommend",
    "such a dreadful {noun}",
]
NOUNS = ["movie", "product", "book", "show", "restaurant", "song", "service"]

torch.manual_seed(42)
def generate_dataset(n_per_class=500):
    texts, labels = [], []
    for _ in range(n_per_class):
        n = torch.randint(0, len(NOUNS), (1,)).item()
        t = torch.randint(0, len(POSITIVE_TEMPLATES), (1,)).item()
        texts.append(POSITIVE_TEMPLATES[t].format(noun=NOUNS[n]))
        labels.append(1)

        n = torch.randint(0, len(NOUNS), (1,)).item()
        t = torch.randint(0, len(NEGATIVE_TEMPLATES), (1,)).item()
        texts.append(NEGATIVE_TEMPLATES[t].format(noun=NOUNS[n]))
        labels.append(0)
    return texts, labels

texts, labels = generate_dataset(800)
# Shuffle
perm = torch.randperm(len(texts)).tolist()
texts  = [texts[i] for i in perm]
labels = [labels[i] for i in perm]

print('Total samples:', len(texts))
print('Positive:', sum(labels), ' Negative:', len(labels) - sum(labels))
print('\nSample positives:')
for t in [x for x, y in zip(texts, labels) if y == 1][:3]:
    print('  ', t)
print('Sample negatives:')
for t in [x for x, y in zip(texts, labels) if y == 0][:3]:
    print('  ', t)

In [ ]:
# Build vocabulary from the corpus
def build_vocab(texts, min_freq=1):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for tok, cnt in counter.most_common():
        if cnt >= min_freq:
            vocab[tok] = len(vocab)
    return vocab

vocab = build_vocab(texts)
print('Vocabulary size:', len(vocab))
print('First 15 entries:', list(vocab.items())[:15])

In [ ]:
# Dataset + collate with padding
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def encode(self, text):
        return [self.vocab.get(tok, self.vocab['<UNK>']) for tok in tokenize(text)]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        ids = torch.tensor(self.encode(self.texts[i]), dtype=torch.long)
        return ids, self.labels[i]

def collate_fn(batch):
    seqs, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, lengths, torch.tensor(labels, dtype=torch.long)

# Split train / val
n_train = int(0.8 * len(texts))
train_ds = SentimentDataset(texts[:n_train], labels[:n_train], vocab)
val_ds   = SentimentDataset(texts[n_train:], labels[n_train:], vocab)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, collate_fn=collate_fn)

print('Train:', len(train_ds), ' Val:', len(val_ds))
xb, lens, yb = next(iter(train_loader))
print('Batch shape:', xb.shape, ' Lengths:', lens[:5], ' Labels:', yb[:5])

In [ ]:
# LSTM sentiment classifier
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_size=64, num_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        self.drop  = nn.Dropout(0.2)
        self.fc    = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        emb = self.embed(x)                                    # [B, T, E]
        packed = pack_padded_sequence(emb, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        out, (h, c) = self.lstm(packed)
        h = h[-1]                                              # [B, H]
        h = self.drop(h)
        return self.fc(h)

model = SentimentLSTM(vocab_size=len(vocab)).to(device)
print(model)
print('Params:', sum(p.numel() for p in model.parameters()))

In [ ]:
# Training helpers
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for xb, lengths, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb, lengths)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for xb, lengths, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb, lengths)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += xb.size(0)
    return total_loss / total, correct / total

torch.manual_seed(42)
model = SentimentLSTM(vocab_size=len(vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 10
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
t0 = time.time()
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
    history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)
    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'train_loss={tr_loss:.4f}  train_acc={tr_acc:.4f} | '
          f'val_loss={va_loss:.4f}  val_acc={va_acc:.4f}')
print(f'\nTotal time: {time.time() - t0:.1f}s')

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'],   label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['train_acc'], label='train')
axes[1].plot(history['val_acc'],   label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Inference on new sentences
def predict(text, model, vocab, device='cpu'):
    model.eval()
    ids = [vocab.get(tok, vocab['<UNK>']) for tok in tokenize(text)]
    if not ids:
        return 'empty'
    xb = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    lengths = torch.tensor([len(ids)])
    with torch.no_grad():
        logits = model(xb, lengths)
        probs  = torch.softmax(logits, dim=1)[0]
    label = 'positive' if probs[1] > probs[0] else 'negative'
    return label, probs.cpu().numpy().round(3)

test_sentences = [
    "i love this movie",
    "this product is terrible",
    "absolutely wonderful show",
    "worst book ever",
    "the restaurant was fantastic",
    "the service was horrible",
]

print(f'{"Sentence":<35} {"Prediction":<10} {"Probs [neg, pos]"}')
print('-' * 72)
for s in test_sentences:
    label, probs = predict(s, model, vocab, device)
    print(f'{s:<35} {label:<10} {probs}')

---
## 🚨 12. Common NLP Mistakes

### ❌ 1. Treating token IDs as meaningful numbers
Token ID `82` is **not** "more" than `15` — it's just an index. Always use **embeddings**.

### ❌ 2. Ignoring padding
The model will treat `<PAD>` as real content unless you mask it.

### ❌ 3. Wrong attention masks
Incorrect masks silently corrupt attention. Double-check that padding positions are masked.

### ❌ 4. Data leakage
Same document or near-duplicate text appearing in both train and test.

### ❌ 5. Removing useful words during preprocessing
Over-aggressive stopword removal can strip **negations** ("not", "never") which flip sentiment.

### ❌ 6. Using accuracy on imbalanced data
Use **F1** or per-class precision/recall.

### ❌ 7. Training a large model from scratch on tiny data
Use **pretrained models** — the same lesson as Module 11.

### ❌ 8. Forgetting tokenization is part of the model
The **same tokenizer** must be used at training and inference.

### ❌ 9. Different preprocessing at inference
Any difference between training and production preprocessing degrades predictions.

---
## 🆚 13. NLP vs Computer Vision

| Aspect | Computer Vision | NLP |
| --- | --- | --- |
| Input | Images / video | Text |
| Basic representation | Pixels / tensors | Tokens / embeddings |
| Common models | CNNs, ViTs | RNNs, Transformers |
| Major challenges | Spatial information | Context / sequence |
| Example | Object detection | Sentiment analysis |

### Key difference

- **Images** have a natural grid structure — CNNs exploit it.
- **Text** has a **sequence** structure — RNNs / Transformers exploit it.

Both fields benefit enormously from **transfer learning** with pretrained models.

---
## 🏋️ 14. Hands-On Exercises

### Exercise 1 — Replace LSTM with GRU
Swap `nn.LSTM` for `nn.GRU`. Compare accuracy, training time, and parameter count.

### Exercise 2 — Bidirectional LSTM
Use `bidirectional=True` and adjust the classifier input size (hidden × 2). Does it help?

### Exercise 3 — Pretrained Embeddings
Load GloVe or Word2Vec vectors, initialize the embedding layer, and freeze it. Compare.

### Exercise 4 — Real Dataset
Download **IMDb** or **SST-2** and train on real text. Report accuracy and F1.

### Exercise 5 — Pretrained Transformer
Fine-tune `distilbert-base-uncased` with HuggingFace `transformers`:
```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
```
Compare to the LSTM baseline on accuracy, params, and speed.

### Exercise 6 — Add Attention
Use `output` (all hidden states) and apply additive attention. Pool with attention weights instead of `hidden[-1]`.

### Exercise 7 — Multi-Class Classification
Extend the task to 3 or 4 classes (e.g. very negative → very positive). Adjust the head and loss.

---
## 🎤 15. Interview Questions

1. **What is NLP?**  
   Natural Language Processing — the field of AI focused on enabling computers to understand and generate human language.

2. **Why can't neural networks directly process raw text?**  
   Text is symbolic — it must be converted into numerical representations (token IDs → embeddings).

3. **What is tokenization?**  
   Splitting text into smaller units (tokens) — words, subwords, or characters.

4. **What is an embedding?**  
   A learned dense vector representation of a token.

5. **Why are LSTMs useful for NLP?**  
   They handle sequences and long-range dependencies better than vanilla RNNs.

6. **What problem does attention solve?**  
   It lets each token directly attend to any other token — solving long-range dependency and context problems.

7. **Why is padding required?**  
   To make sequences in a batch the same length so they can be processed as a tensor.

8. **What is an attention mask?**  
   A binary mask that tells the model which positions are real content and which are padding.

9. **Classification vs sequence-to-sequence?**  
   Classification → one label per input.  
   Seq2seq → one sequence per input sequence.

10. **Why did Transformers become important for NLP?**  
    Direct token-to-token attention + parallel training + scalability → state-of-the-art performance across many tasks.

---
## ✅ 16. Knowledge Check

- [ ] Explain what NLP is
- [ ] Explain why text needs numerical representations
- [ ] Describe tokenization
- [ ] Explain word vs subword tokenization
- [ ] Use `nn.Embedding` correctly
- [ ] Describe how RNN / LSTM processes sequences
- [ ] Explain what attention solves
- [ ] Explain padding and attention masks
- [ ] Distinguish classification vs seq2seq
- [ ] Choose appropriate metrics (F1, BLEU, ROUGE)
- [ ] Build and train a sentiment classifier in PyTorch

---
## 🏁 17. Key Takeaways

### The NLP pipeline

```text
Text
 ↓
Tokens
 ↓
Token IDs
 ↓
Embeddings
 ↓
Sequence Model (RNN/LSTM/Transformer)
 ↓
Prediction
```

### The evolution

```text
RNN
 ↓
LSTM / GRU
 ↓
Attention
 ↓
Transformer
 ↓
LLMs
```

### The critical rules

> **Text is not numerical — always embed tokens.**  
> **Pad and mask variable-length sequences.**  
> **Use the same tokenizer and preprocessing at inference.**  
> **Never use accuracy alone on imbalanced data.**  
> **Use pretrained models when data is scarce.**

### 🧭 Golden Rule

> **NLP is sequence modeling with context. Tokens → embeddings → attention is the path from raw text to meaning.**

---

### 🔗 What's Next?

**Module 16 — Model Deployment**

We've **built** models. Now we'll **ship** them — from saving `state_dict` and `model.eval()` to FastAPI endpoints, Docker containers, and production considerations.